# generte postionnal encoding matrix PE 

In [16]:
from positional_enoding import generate_postionnal_encoding_matrix
import numpy as np
from torch.nn import Softmax
tokenised_texte = ["the" , "cat" , "is" , "so" ]
PE_matrix = generate_postionnal_encoding_matrix(tokenised_texte=tokenised_texte , d_model= 4)
print(PE_matrix)
print(np.shape(PE_matrix))


[[0.0, 1.0, 0.0, 1.0], [0.8414709848078965, 0.5403023058681398, 0.009999833334166664, 0.9999500004166653], [0.9092974268256817, -0.4161468365471424, 0.01999866669333308, 0.9998000066665778], [0.1411200080598672, -0.9899924966004454, 0.02999550020249566, 0.9995500337489875]]
(4, 4)


# Create the embeding Matrix Xe

In [18]:

np.random.default_rng(42)
d_model = 4
Xe = np.random.rand(d_model,d_model)
Wq = np.random.rand(d_model,d_model)
Wk = np.random.rand(d_model,d_model)
Wv = np.random.rand(d_model,d_model)
X_embeding = Xe + PE_matrix
print(X_embeding)


[[ 0.06305617  1.97745946  0.34369256  1.25172501]
 [ 1.49542561  1.09384357  0.40955639  1.71995869]
 [ 1.10940686 -0.05089815  0.32835947  1.58460366]
 [ 0.64593842 -0.05385033  0.52923731  1.00811918]]


In [19]:
import torch
from torch import tensor

def attention(Xe,Wq,Wk,Wv,d_model):
    
    if Xe.shape[0] != Wq.shape[1] :
        raise Exception('dimension not correct')

    Q = np.matmul(Xe,Wq)

    if Xe.shape[0] != Wk.shape[1] :
            raise Exception('dimension not correct')

    K = np.matmul(Xe,Wk)

    if Xe.shape[0] != Wv.shape[1] :
            raise Exception('dimension not correct')

    V = np.matmul(Xe,Wv)

    Attention = np.matmul(Q,K.T)
    Attention = Attention / np.sqrt(d_model)
    Attention = tensor(Attention)
    softmax = Softmax(dim=-1)
    Attention = softmax(Attention)
    Attention = Attention.numpy()
    #Attention = np.matmul(Attention , V)

    return Attention

In [13]:
A = attention(Xe,Wq,Wk,Wv,d_model)
print(A)


[[0.20486632 0.10051147 0.14883945 0.24576515 0.30001762]
 [0.20650159 0.11666268 0.15514477 0.23102525 0.29066572]
 [0.20410982 0.10467534 0.15271897 0.2511601  0.28733577]
 [0.2031815  0.0947066  0.14556038 0.25333248 0.30321905]
 [0.20579926 0.09556186 0.14430791 0.23834481 0.31598616]]


In [20]:
WO = np.random.rand(d_model , d_model)
def layer_normalisation(X):
    epsilon = 10**-4
    mean = np.mean(X)
    std = np.std(X)
    return (X - mean) / (np.sqrt((std**2 + epsilon) ) )

def multi_head_projection(V , WO):
    Z = np.matmul(A,V)
    Y_att = np.matmul(Z,WO)
    return  Y_att

def first_risdual_addition(X_in,V,WO):
      Y_attn = multi_head_projection(V,WO)
      X_1 = X_in + Y_attn
      X_1 = layer_normalisation(d_model,d_model)
      return X_1

def postion_wise_feed_forward(X,W1,W2,bias1,bias2):
     X = np.matmul(X,W1) + bias1
     tensor_X = torch.tensor(X)
     tensor_X = torch.nn.functional.relu(tensor_X)
     X = tensor_X.numpy()
     Y_ffn = np.matmul(X,W2) + bias2
     return Y_ffn


def Encoder_output(Y_ffn):
     Y_ffn = X_embeding + Y_ffn
     H_encoder = layer_normalisation(Y_ffn)
     return H_encoder


def Encoder_operations(dff):

    # the dff is changing the dimension of projection then reproject it into the orginal projection dimension
     V = np.matmul(Xe,Wv)
     bias1 = np.random.rand(d_model)
     bias2 = np.random.rand(d_model)
     W1 = np.random.rand(d_model,dff)
     W2 = np.random.rand(dff,d_model)
     Y_att = multi_head_projection(V,WO)
     X_1 = layer_normalisation(Xe) + layer_normalisation(Y_att)
     Y_ffn = postion_wise_feed_forward(X_1,W1,W2,bias1,bias2)
     H_encoder = Encoder_output(Y_ffn)
     return H_encoder


H_encoder = Encoder_output(3)
print(H_encoder)


[[-1.22880511  1.79518924 -0.78551144  0.64881799]
 [ 1.03376784  0.39942824 -0.68147284  1.38844063]
 [ 0.42401203 -1.40880753 -0.80973162  1.17463362]
 [-0.30808342 -1.4134708  -0.49242468  0.26401786]]
